# 03 · Gold contract tables (Spark SQL)

The same grain and columns as Project 1's gold tables, so the Power BI model can read either producer.
`fact_sales` carries the promotion baseline (28-day trailing non-promo average over trading days);
`fact_stockout_risk` is Project 1's single-day rule (λ ≥ 3, full trading day, resumes within 28 days).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../src"))  # the repo's src/, via the Databricks Git folder

from favorita_spark.config import Config

cfg = Config()
spark.sql(f"USE CATALOG {cfg.catalog}")

In [ ]:
from favorita_spark.runner import build_sql_layer

build_sql_layer(spark, "gold", cfg)

In [ ]:
%sql
-- single-day stock-out flags and estimated lost units per month
SELECT d.year, d.month_num, count(*) AS flagged_days, round(sum(r.expected_units)) AS est_lost_units
FROM gold.fact_stockout_risk AS r
JOIN gold.dim_date AS d USING (date_key)
GROUP BY d.year, d.month_num
ORDER BY d.year, d.month_num